# 폐기물 이미지 분류: Keras vs PyTorch CNN 프레임워크 비교

**연구 질문**: 동일한 CNN 구조와 동일한 데이터셋을 사용할 때, 딥러닝 프레임워크(Keras vs PyTorch)에 따른 성능 차이가 발생하는가?

**데이터셋**: TrashNet (garythung/trashnet, Hugging Face) — 종이, 유리, 금속, 플라스틱, 일반쓰레기 5종 (원본 6종 중 cardboard를 paper로 통합)

**진행 순서**
1. 동일 데이터셋 준비
2. 동일 CNN 구조 설계
3. Keras로 학습
4. PyTorch로 학습
5. 클래스별 성능 비교
6. 전체 성능 비교
7. Confusion Matrix 비교
8. 학습/추론 속도 비교


### [1] 라이브러리 설치 및 불러오기

In [ ]:
!pip install -q "datasets==2.19.1"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("TensorFlow 버전:", tf.__version__)
print("PyTorch 버전:", torch.__version__)
print("TF GPU 사용 가능:", tf.config.list_physical_devices('GPU'))
print("PyTorch GPU 사용 가능:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch 사용 디바이스:", device)

### [1-1] 한글 폰트 설정 (Colab)

Colab 기본 matplotlib에는 한글 폰트가 없어 그래프의 한글이 네모(□)로 깨집니다. 나눔고딕 폰트를 설치하고 matplotlib에 등록합니다. **설치 후 아래 셀을 한 번 더 실행(Ctrl+Enter)하면 폰트가 정상 적용됩니다.**

In [ ]:
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1

import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# 새로 등록한 폰트가 바로 적용되지 않으면 캐시를 지우고 런타임을 재시작하세요.
# (메뉴: 런타임 > 런타임 다시 시작)
print("한글 폰트 설정 완료:", plt.rcParams["font.family"])

### [2] 데이터셋 다운로드 (TrashNet)

TrashNet은 종이(paper), 유리(glass), 금속(metal), 플라스틱(plastic), 골판지(cardboard), 일반쓰레기(trash) 6개 클래스, 약 5,000장의 실제 쓰레기 사진으로 구성된 데이터셋입니다. Hugging Face `datasets` 라이브러리로 바로 불러올 수 있습니다.

**참고**: 이 셀에서 `ValueError: Invalid string class label ...@<커밋해시>` 오류가 나면, `datasets` 라이브러리 최신 버전이 이 데이터셋의 로딩 스크립트를 처리할 때 생기는 버그일 가능성이 높습니다. 이미 [1]번 셀에서 `datasets==2.19.1`로 버전을 고정해뒀으니, 런타임을 재시작(런타임 → 런타임 다시 시작)한 뒤 [1]번 셀부터 다시 실행해보세요. 그래도 동일한 오류가 나면 `load_dataset("garythung/trashnet", download_mode="force_redownload")`로 캐시를 강제로 새로 받아보는 것도 시도해볼 수 있습니다.

In [ ]:
from datasets import load_dataset

print("TrashNet 데이터셋 다운로드 중...")
raw_dataset = load_dataset("garythung/trashnet")
print(raw_dataset)

original_class_names = raw_dataset["train"].features["label"].names
print("원본 클래스:", original_class_names)

### [3] 5개 클래스로 재구성

과제 목표에 맞춰 `cardboard`를 `paper`(종이) 카테고리로 통합하여 종이/유리/금속/플라스틱/일반쓰레기 5개 클래스로 재구성합니다.

In [ ]:
# 종이류(cardboard)를 종이(paper) 카테고리로 통합하여 5개 클래스 구성
merge_map = {
    "cardboard": "paper",
    "glass": "glass",
    "metal": "metal",
    "paper": "paper",
    "plastic": "plastic",
    "trash": "trash",
}

class_names_kor = {
    "paper": "종이",
    "glass": "유리",
    "metal": "금속",
    "plastic": "플라스틱",
    "trash": "일반쓰레기",
}

new_class_names = ["paper", "glass", "metal", "plastic", "trash"]
class_to_idx = {name: i for i, name in enumerate(new_class_names)}

def remap_label(example):
    original_name = original_class_names[example["label"]]
    new_name = merge_map[original_name]
    example["label"] = class_to_idx[new_name]
    return example

dataset = raw_dataset["train"].map(remap_label)
print("재구성된 클래스:", new_class_names)

### [4] 이미지 배열 변환 및 train/val/test 분할

In [ ]:
IMG_SIZE = 128

def preprocess_to_array(hf_dataset, img_size=IMG_SIZE):
    images = []
    labels = []
    for example in hf_dataset:
        img = example["image"].convert("RGB").resize((img_size, img_size))
        images.append(np.array(img, dtype=np.float32) / 255.0)
        labels.append(example["label"])
    return np.array(images), np.array(labels)

print("이미지 배열 변환 중... (수 분 소요될 수 있습니다)")
X, y = preprocess_to_array(dataset)
print("전체 이미지 shape:", X.shape)
print("클래스별 이미지 수:")
for i, name in enumerate(new_class_names):
    print(f"  {class_names_kor[name]}({name}): {np.sum(y==i)}장")

# train(70%) / val(15%) / test(15%) 분할, 클래스 비율 유지(stratify)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"\n학습 데이터: {X_train.shape[0]}장")
print(f"검증 데이터: {X_val.shape[0]}장")
print(f"테스트 데이터: {X_test.shape[0]}장")

### [4-1] 클래스 가중치 계산 (클래스 불균형 보정)

학습 데이터의 클래스별 개수를 기준으로 가중치를 계산합니다. 개수가 적은 클래스일수록(예: 일반쓰레기) 더 큰 가중치를 부여해, 손실 계산 시 소수 클래스를 더 신경 쓰도록 유도합니다. 단순 역수(전체/개수)로 계산하면 가중치가 지나치게 커져(예: 일반쓰레기 약 3.68배) 다수 클래스 성능을 과도하게 희생시킬 수 있어, **제곱근을 취해 완화**한 값을 사용합니다. **Keras와 PyTorch가 정확히 동일한 가중치**를 사용하도록, 하나의 계산식으로 두 프레임워크가 쓸 값을 함께 만듭니다.

In [ ]:
train_counts = Counter(y_train)
n_classes = len(new_class_names)
total_train = len(y_train)

# 기본 가중치 = 전체 샘플 수 / (클래스 수 * 해당 클래스 샘플 수)
raw_weights = {
    i: total_train / (n_classes * train_counts[i]) for i in range(n_classes)
}
# 가중치가 너무 극단적으로 커지지 않도록 제곱근으로 완화
class_weights_dict = {i: float(np.sqrt(raw_weights[i])) for i in range(n_classes)}

print("클래스별 가중치 (완화 전 -> 완화 후):")
for i in range(n_classes):
    print(f"  {class_names_kor[new_class_names[i]]}: {raw_weights[i]:.3f} -> {class_weights_dict[i]:.3f}")

# PyTorch 손실함수에 바로 넣을 수 있는 tensor 형태
class_weights_tensor = torch.tensor(
    [class_weights_dict[i] for i in range(n_classes)], dtype=torch.float32
)

### [5] 데이터 시각화 (클래스 분포 & 샘플 이미지)

In [ ]:
counts = [int(np.sum(y == i)) for i in range(len(new_class_names))]
colors = ['#B85042', '#A7BEAE', '#7A9CC6', '#E7A34C', '#6D2E46']

plt.figure(figsize=(7, 4))
plt.bar([class_names_kor[n] for n in new_class_names], counts, color=colors)
plt.title("클래스별 이미지 수")
plt.ylabel("이미지 수")
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.show()

plt.figure(figsize=(15, 4))
for i, name in enumerate(new_class_names):
    idx = np.where(y_train == i)[0][0]
    ax = plt.subplot(1, 5, i + 1)
    plt.imshow(X_train[idx])
    plt.title(class_names_kor[name], fontsize=14, fontweight="bold")
    plt.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#CCCCCC")
        spine.set_linewidth(1.5)
plt.tight_layout()
plt.savefig("sample_images.png", dpi=200, bbox_inches="tight")
plt.show()

### [6] 공통 CNN 구조 설계

프레임워크 비교의 핵심 전제 조건은 **완전히 동일한 구조**를 사용하는 것입니다. Keras와 PyTorch 모두 아래 구조를 그대로 구현합니다.

| Layer | 세부 사항 |
|---|---|
| Conv Block 1 | Conv2D(32, 3x3) → ReLU → MaxPool(2x2) |
| Conv Block 2 | Conv2D(64, 3x3) → ReLU → MaxPool(2x2) |
| Conv Block 3 | Conv2D(128, 3x3) → ReLU → MaxPool(2x2) |
| Flatten | - |
| Dense | 128 → ReLU → Dropout(0.5) |
| Output | Dense(5) → Softmax |

입력 크기: 128×128×3, 클래스 수: 5, Optimizer: Adam, Loss: Cross-Entropy, Epochs: 30(최대), Batch size: 32

**정확도 개선을 위해 다음 4가지 학습 기법을 Keras·PyTorch에 동일하게 추가합니다** (모델 구조 자체는 바뀌지 않으므로 파라미터 수는 여전히 동일합니다).

- **데이터 증강(Augmentation)**: 좌우 반전 + 회전(±18°) + 확대/축소(±10%)를 학습 데이터에만 적용
- **클래스 가중치(Class Weight)**: 위 [4-1]에서 계산한(제곱근으로 완화된) 가중치를 학습 손실에만 적용해 소수 클래스(일반쓰레기 등)를 너무 세지 않게 신경 쓰도록 함
- **Early Stopping**: 검증 손실이 5epoch 연속 개선되지 않으면 학습을 멈추고, 가장 좋았던 시점의 가중치로 복원
- **최대 Epoch 확장 (20 → 30)**: 증강으로 학습이 더 어려워진 만큼 더 오래 학습할 여지를 줌 (Early Stopping이 있어 불필요하게 길어지지는 않음)

### [7] Keras CNN 모델 구축

입력 직후에 데이터 증강 레이어(RandomFlip, RandomRotation, RandomZoom)를 추가합니다. 이 레이어들은 `model.fit()` 학습 중에만 동작하고, `predict()`/`evaluate()` 시에는 자동으로 비활성화되므로 평가에는 영향을 주지 않습니다. 학습 파라미터가 없는 레이어라 전체 파라미터 수도 그대로 유지됩니다.

In [ ]:
def build_keras_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=5):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        # 데이터 증강 (학습 시에만 적용됨)
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

keras_model = build_keras_cnn()
keras_model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])
keras_model.summary()

### [8] Keras 모델 학습 (학습 시간 측정)

`class_weight`로 클래스 불균형을 보정하고, `EarlyStopping`으로 과적합을 방지합니다. `restore_best_weights=True`로 설정해, 학습이 끝나면 검증 손실이 가장 낮았던 시점의 가중치가 자동으로 복원됩니다.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

keras_start_time = time.time()
keras_history = keras_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_dict,
    callbacks=[early_stop],
    verbose=1
)
keras_train_time = time.time() - keras_start_time
print(f"\nKeras 학습 소요 시간: {keras_train_time:.2f}초")

### [9] Keras 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(keras_history.history['accuracy'], label='train')
axes[0].plot(keras_history.history['val_accuracy'], label='val')
axes[0].set_title('Keras 정확도')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(keras_history.history['loss'], label='train')
axes[1].plot(keras_history.history['val_loss'], label='val')
axes[1].set_title('Keras 손실')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig("keras_training_curves.png")
plt.show()

### [10] Keras 테스트 평가 및 추론 속도 측정

In [ ]:
keras_infer_start = time.time()
keras_pred_probs = keras_model.predict(X_test, batch_size=BATCH_SIZE)
keras_infer_time = time.time() - keras_infer_start
keras_pred = np.argmax(keras_pred_probs, axis=1)

keras_test_acc = accuracy_score(y_test, keras_pred)
print(f"Keras 테스트 정확도: {keras_test_acc:.4f}")
print(f"Keras 추론 소요 시간(테스트셋 전체): {keras_infer_time:.4f}초")
print(f"Keras 이미지당 평균 추론 시간: {(keras_infer_time / len(X_test)) * 1000:.4f} ms")

target_names_kor = [class_names_kor[n] for n in new_class_names]

print("\n[Keras] 클래스별 분류 리포트")
keras_report = classification_report(
    y_test, keras_pred, target_names=target_names_kor, output_dict=True)
print(classification_report(y_test, keras_pred, target_names=target_names_kor))

### [11] Keras Confusion Matrix

In [ ]:
keras_cm = confusion_matrix(y_test, keras_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(keras_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_kor, yticklabels=target_names_kor)
plt.title("Keras Confusion Matrix")
plt.xlabel("예측"); plt.ylabel("실제")
plt.tight_layout()
plt.savefig("keras_confusion_matrix.png")
plt.show()

### [12] PyTorch Dataset / DataLoader 구축

이전 버전에서는 증강(좌우 반전, 회전, 확대/축소)을 `torchvision.transforms`로 이미지 한 장씩 CPU에서 처리했습니다. Keras는 증강 레이어가 모델 안에 포함되어 GPU 연산 그래프에서 배치 전체를 한 번에 처리하는 것과 달리, 이 방식은 구조적으로 불리했습니다. 그래서 이번에는 원본 텐서를 그대로 GPU에 올리고, **증강은 학습 루프 안에서 배치 단위로 GPU 위에서** 수행하도록 바꿉니다. Validation/Test 데이터에는 증강을 적용하지 않아 Keras와 조건을 동일하게 맞춥니다.

In [ ]:
class TrashDataset(Dataset):
    def __init__(self, images, labels):
        # (N, H, W, C) -> (N, C, H, W): PyTorch의 채널 순서에 맞게 변환
        self.images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = TrashDataset(X_train, y_train)
val_dataset = TrashDataset(X_val, y_val)
test_dataset = TrashDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("PyTorch DataLoader 준비 완료")

### [12-1] GPU 배치 증강 함수

`torch.nn.functional.affine_grid` + `grid_sample`을 사용해, 배치 전체에 대해 샘플마다 서로 다른 회전·확대축소·좌우반전을 **GPU 위에서 한 번의 텐서 연산으로** 적용합니다. 외부 라이브러리(kornia 등) 없이 PyTorch 기본 기능만으로 구현했고, Keras의 RandomFlip/RandomRotation/RandomZoom 레이어와 각도·비율 범위를 동일하게(±18°, ±10%) 맞췄습니다.

In [ ]:
import torch.nn.functional as F
import math

def gpu_augment(batch, degrees=18, scale_range=(0.9, 1.1), flip_prob=0.5):
    """배치 텐서(N,C,H,W)에 샘플별 랜덤 회전/확대축소/좌우반전을 GPU에서 한 번에 적용"""
    N = batch.size(0)
    device = batch.device
    max_rad = math.radians(degrees)

    angles = (torch.rand(N, device=device) * 2 - 1) * max_rad
    scales = torch.rand(N, device=device) * (scale_range[1] - scale_range[0]) + scale_range[0]
    flips = torch.where(torch.rand(N, device=device) < flip_prob, -1.0, 1.0)

    cos = torch.cos(angles) * scales
    sin = torch.sin(angles) * scales
    theta = torch.zeros(N, 2, 3, device=device)
    theta[:, 0, 0] = cos * flips
    theta[:, 0, 1] = -sin
    theta[:, 1, 0] = sin * flips
    theta[:, 1, 1] = cos

    grid = F.affine_grid(theta, batch.size(), align_corners=False)
    return F.grid_sample(batch, grid, align_corners=False, padding_mode='zeros')

### [13] PyTorch CNN 모델 정의 (Keras와 동일 구조)

In [ ]:
class TrashCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

        flat_size = self._get_flat_size()
        self.fc1 = nn.Linear(flat_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def _get_flat_size(self):
        with torch.no_grad():
            x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            x = self.pool(self.relu(self.conv1(x)))
            x = self.pool(self.relu(self.conv2(x)))
            x = self.pool(self.relu(self.conv3(x)))
            return x.view(1, -1).shape[1]

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

pytorch_model = TrashCNN().to(device)
print(pytorch_model)

total_params = sum(p.numel() for p in pytorch_model.parameters())
print(f"\nPyTorch 모델 전체 파라미터 수: {total_params:,}")
print(f"Keras 모델 전체 파라미터 수: {keras_model.count_params():,}")

### [14] PyTorch 모델 학습 (학습 시간 측정)

클래스 가중치는 **학습 손실 계산에만** 적용합니다(`criterion`). 검증/평가용 손실은 가중치 없는 `criterion_eval`을 별도로 사용해, Keras의 `class_weight`가 검증 손실에는 영향을 주지 않는 동작과 동일하게 맞춥니다. Early Stopping은 검증 손실 기준 patience=5로 Keras와 동일하게 설정합니다.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))  # 학습용(가중치 적용)
criterion_eval = nn.CrossEntropyLoss()  # 검증/평가용(가중치 없음, Keras와 동일 조건)
optimizer = optim.Adam(pytorch_model.parameters())

def evaluate_loader(model, loader, loss_fn):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss_sum += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total

pytorch_history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

PATIENCE = 5
best_val_loss = float('inf')
best_state = None
patience_counter = 0

pytorch_start_time = time.time()
for epoch in range(EPOCHS):
    pytorch_model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = gpu_augment(images)  # 학습 배치에만 GPU 증강 적용
        optimizer.zero_grad()
        outputs = pytorch_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total
    val_loss, val_acc = evaluate_loader(pytorch_model, val_loader, criterion_eval)

    pytorch_history["loss"].append(train_loss)
    pytorch_history["accuracy"].append(train_acc)
    pytorch_history["val_loss"].append(val_loss)
    pytorch_history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {train_loss:.4f} - acc: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

    # Early Stopping (검증 손실 기준)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(pytorch_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping: {epoch+1} epoch에서 학습을 종료합니다 (best val_loss: {best_val_loss:.4f})")
            break

# 검증 손실이 가장 낮았던 시점의 가중치로 복원 (Keras의 restore_best_weights=True와 동일)
if best_state is not None:
    pytorch_model.load_state_dict(best_state)

pytorch_train_time = time.time() - pytorch_start_time
print(f"\nPyTorch 학습 소요 시간: {pytorch_train_time:.2f}초")

### [15] PyTorch 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pytorch_history['accuracy'], label='train')
axes[0].plot(pytorch_history['val_accuracy'], label='val')
axes[0].set_title('PyTorch 정확도')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(pytorch_history['loss'], label='train')
axes[1].plot(pytorch_history['val_loss'], label='val')
axes[1].set_title('PyTorch 손실')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig("pytorch_training_curves.png")
plt.show()

### [15-1] Keras vs PyTorch 학습 곡선 직접 비교

발표자료의 **'Accuracy 비교' / 'Loss 비교'** 슬라이드에는 아래 두 그래프를 사용하세요.

In [ ]:
# Accuracy 비교 (검증 정확도 기준)
plt.figure(figsize=(8, 5))
plt.plot(keras_history.history['val_accuracy'], label='Keras', color='#7A9CC6', linewidth=2)
plt.plot(pytorch_history['val_accuracy'], label='PyTorch', color='#E7A34C', linewidth=2)
plt.title('Keras vs PyTorch - Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.tight_layout()
plt.savefig("accuracy_comparison.png")
plt.show()

In [ ]:
# Loss 비교 (검증 손실 기준)
plt.figure(figsize=(8, 5))
plt.plot(keras_history.history['val_loss'], label='Keras', color='#7A9CC6', linewidth=2)
plt.plot(pytorch_history['val_loss'], label='PyTorch', color='#E7A34C', linewidth=2)
plt.title('Keras vs PyTorch - Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.tight_layout()
plt.savefig("loss_comparison.png")
plt.show()

### [16] PyTorch 테스트 평가 및 추론 속도 측정

In [ ]:
pytorch_model.eval()
all_preds, all_labels = [], []

pytorch_infer_start = time.time()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = pytorch_model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
pytorch_infer_time = time.time() - pytorch_infer_start

pytorch_pred = np.array(all_preds)
pytorch_test_acc = accuracy_score(y_test, pytorch_pred)

print(f"PyTorch 테스트 정확도: {pytorch_test_acc:.4f}")
print(f"PyTorch 추론 소요 시간(테스트셋 전체): {pytorch_infer_time:.4f}초")
print(f"PyTorch 이미지당 평균 추론 시간: {(pytorch_infer_time / len(X_test)) * 1000:.4f} ms")

print("\n[PyTorch] 클래스별 분류 리포트")
pytorch_report = classification_report(
    y_test, pytorch_pred, target_names=target_names_kor, output_dict=True)
print(classification_report(y_test, pytorch_pred, target_names=target_names_kor))

### [17] PyTorch Confusion Matrix

In [ ]:
pytorch_cm = confusion_matrix(y_test, pytorch_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(pytorch_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=target_names_kor, yticklabels=target_names_kor)
plt.title("PyTorch Confusion Matrix")
plt.xlabel("예측"); plt.ylabel("실제")
plt.tight_layout()
plt.savefig("pytorch_confusion_matrix.png")
plt.show()

### [18] 전체 성능 비교: Keras vs PyTorch

In [ ]:
comparison_df = pd.DataFrame({
    "지표": ["테스트 정확도", "학습 시간(초)", "전체 추론 시간(초)",
            "이미지당 추론 시간(ms)", "파라미터 수"],
    "Keras": [
        round(keras_test_acc, 4), round(keras_train_time, 2), round(keras_infer_time, 4),
        round((keras_infer_time / len(X_test)) * 1000, 4), keras_model.count_params(),
    ],
    "PyTorch": [
        round(pytorch_test_acc, 4), round(pytorch_train_time, 2), round(pytorch_infer_time, 4),
        round((pytorch_infer_time / len(X_test)) * 1000, 4), total_params,
    ],
})
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

frameworks = ['Keras', 'PyTorch']
colors_fw = ['#7A9CC6', '#E7A34C']

axes[0].bar(frameworks, [keras_test_acc, pytorch_test_acc], color=colors_fw)
axes[0].set_title("테스트 정확도")
axes[0].set_ylim(0, 1)

axes[1].bar(frameworks, [keras_train_time, pytorch_train_time], color=colors_fw)
axes[1].set_title("학습 시간 (초)")

axes[2].bar(frameworks,
            [(keras_infer_time / len(X_test)) * 1000, (pytorch_infer_time / len(X_test)) * 1000],
            color=colors_fw)
axes[2].set_title("이미지당 추론 시간 (ms)")

plt.tight_layout()
plt.savefig("framework_comparison.png")
plt.show()

### [19] 클래스별 Precision / Recall / F1-score 비교

발표자료의 **'클래스별 Precision/Recall/F1 비교'** 슬라이드에는 아래 표(class_metrics_df)와 그래프(class_metrics_comparison.png)를 사용하세요.

In [ ]:
rows = []
for name in target_names_kor:
    rows.append({
        "클래스": name,
        "Keras Precision": keras_report[name]["precision"],
        "Keras Recall": keras_report[name]["recall"],
        "Keras F1": keras_report[name]["f1-score"],
        "PyTorch Precision": pytorch_report[name]["precision"],
        "PyTorch Recall": pytorch_report[name]["recall"],
        "PyTorch F1": pytorch_report[name]["f1-score"],
    })
class_metrics_df = pd.DataFrame(rows).round(4)
class_metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
metrics = [('Precision', 'Keras Precision', 'PyTorch Precision'),
           ('Recall', 'Keras Recall', 'PyTorch Recall'),
           ('F1-score', 'Keras F1', 'PyTorch F1')]
x = np.arange(len(target_names_kor))
width = 0.35

for ax, (title, k_col, p_col) in zip(axes, metrics):
    ax.bar(x - width/2, class_metrics_df[k_col], width, label='Keras', color='#7A9CC6')
    ax.bar(x + width/2, class_metrics_df[p_col], width, label='PyTorch', color='#E7A34C')
    ax.set_xticks(x)
    ax.set_xticklabels(target_names_kor, rotation=20)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.savefig("class_metrics_comparison.png")
plt.show()

### [20] 개별 예측 시각화

교수님 샘플 노트북과 동일한 방식으로, 테스트 샘플에 대한 실제 라벨과 두 프레임워크의 예측 결과를 나란히 시각화합니다.

In [ ]:
np.random.seed(1)
sample_idx = np.random.choice(len(X_test), 9, replace=False)

plt.figure(figsize=(13, 13))
for plot_i, idx in enumerate(sample_idx):
    plt.subplot(3, 3, plot_i + 1)
    plt.imshow(X_test[idx])
    true_label = target_names_kor[y_test[idx]]
    k_pred = target_names_kor[keras_pred[idx]]
    p_pred = target_names_kor[pytorch_pred[idx]]
    k_mark = 'O' if keras_pred[idx] == y_test[idx] else 'X'
    p_mark = 'O' if pytorch_pred[idx] == y_test[idx] else 'X'
    plt.title(f"정답: {true_label}\nKeras: {k_pred} ({k_mark})  |  PyTorch: {p_pred} ({p_mark})",
              fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.savefig("individual_predictions.png")
plt.show()

### [21] 결과 해석 및 결론

아래 항목을 실제 실행 결과(정확도, 학습/추론 시간, 클래스별 F1-score, Confusion Matrix)를 바탕으로 채워 넣어 결론을 작성합니다.

- **정확도 비교**: 동일 구조·동일 데이터에서 Keras와 PyTorch의 테스트 정확도 차이는 (%p) 수준이었다 → 프레임워크 자체보다는 (가중치 초기화, 데이터 셔플링 등) 확률적 요인에 의한 차이인지 확인 필요
- **학습 속도**: Keras가 PyTorch보다 (빠름/느림) — cuDNN 자동 최적화, 그래프 컴파일 방식의 차이 등이 원인으로 추정
- **추론 속도**: 이미지당 평균 추론 시간 비교 결과 (프레임워크)가 더 빠름
- **클래스별 성능**: 유리·금속처럼 반사·재질 특징이 뚜렷한 클래스는 두 프레임워크 모두 F1-score가 높았고, 일반쓰레기처럼 형태가 다양한 클래스는 상대적으로 낮았다
- **결론**: 동일한 CNN 구조와 데이터셋에서는 두 프레임워크의 **분류 성능(정확도, F1-score)** 차이는 미미하지만, **학습/추론 속도**에서는 프레임워크에 따른 차이가 관찰되었다 (또는 관찰되지 않았다)

### [22] 모델 저장

In [ ]:
keras_model.save("waste_classifier_keras.keras")
torch.save(pytorch_model.state_dict(), "waste_classifier_pytorch.pth")
print("Keras 모델 저장 완료: waste_classifier_keras.keras")
print("PyTorch 모델 저장 완료: waste_classifier_pytorch.pth")

### [23] 다중 시드 검증 (재현성 확인용)

**참고**: GPU 증강 적용 전후 결과를 비교해보면, 둘 다 시드는 42로 동일했는데도 PyTorch 정확도가 77.87% → 72.33%로 달라졌습니다. 시드가 같은데도 결과가 달라졌다는 건 두 가지 가능성을 함께 보여줍니다. ① 증강 구현이 실제로 달라진 것(의도한 변화), ② `grid_sample` 같은 연산은 같은 시드를 줘도 GPU 상에서 완전히 결정적으로 재현되지 않을 수 있다는 것(PyTorch의 알려진 특성). 이 둘을 구분하려면, **GPU 증강이 적용된 지금 코드로 시드를 여러 개 바꿔가며 반복 실행**해서 PyTorch 정확도가 매번 72% 근처로 낮게 나오는지, 아니면 다시 77% 대로 돌아오기도 하는지 확인하는 것이 가장 확실합니다. 이제 PyTorch 학습이 약 45초로 빨라졌기 때문에 몇 번 더 돌려보는 데 큰 부담이 없습니다.

**진행 방법**:
1. [1]번 셀에서 `np.random.seed(42)`, `tf.random.set_seed(42)`, `torch.manual_seed(42)`의 42를 다른 숫자(예: 7)로 바꿉니다.
2. 런타임 → 런타임 다시 시작 후, [1]번 셀부터 [18]번 셀까지 순서대로 재실행합니다.
3. [18]번 셀에 나온 Keras/PyTorch 테스트 정확도를 아래 코드에 기록합니다.
4. 시드를 하나 더(예: 123) 바꿔서 2~3을 반복합니다.
5. 아래 셀을 실행해 평균과 표준편차를 확인합니다. PyTorch 값들의 표준편차가 크면(예: 3%p 이상) 이번 72.33%는 정상적인 변동 범위 안에 있는 것이고, 값들이 대부분 낮게 몰려 있다면 GPU 증강 방식 자체가 원인일 가능성이 높습니다.

In [ ]:
import numpy as np

# GPU 증강 적용 버전의 코드로, 시드를 바꿔가며 확인한 [18]번 셀 결과를 여기에 채워 넣으세요
seed_results = {
    42:  {"keras": 0.7826, "pytorch": 0.7233},   # 이번에 실행한 결과
    # 7:   {"keras": None, "pytorch": None},     # 추가로 실행한 뒤 여기에 기록
    # 123: {"keras": None, "pytorch": None},
}

keras_accs = [v["keras"] for v in seed_results.values() if v["keras"] is not None]
pytorch_accs = [v["pytorch"] for v in seed_results.values() if v["pytorch"] is not None]

print(f"Keras   평균: {np.mean(keras_accs):.4f}  표준편차: {np.std(keras_accs):.4f}  (n={len(keras_accs)})")
print(f"PyTorch 평균: {np.mean(pytorch_accs):.4f}  표준편차: {np.std(pytorch_accs):.4f}  (n={len(pytorch_accs)})")

# 참고: GPU 증강 적용 전(CPU 증강, 동일 시드 42) 결과와 비교
print("\n참고: GPU 증강 적용 이전(CPU 증강, seed 42) 결과 — Keras: 0.7708, PyTorch: 0.7787")